In [20]:
import requests as r
import requests
from bs4 import BeautifulSoup as b, Tag
import json

from dataclasses import dataclass
from datetime import datetime, timedelta, time, date
from functools import wraps
import time as t

import sys
import os
import re

# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the parent directory (assuming the notebook is in folder B)
parent_dir = os.path.abspath(os.path.join(notebook_dir, os.pardir))

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from monitoring import strategies as strats
from monitoring.strategies import URL

In [21]:
# load username and password from secrets
QQlogin = ""
QQpassword = ""

with open('secrets.json', 'r', encoding='utf-8') as secs:
    dict = json.load(secs)

    QQlogin = dict["QQlogin"]
    QQpassword = dict["QQpassword"]

    print(f'{hash(QQlogin) = } \n{hash(QQpassword) = } ')

# # Code to write into the secrets file
# secrets = {"QQlogin": "", "QQpassword": ""}
# dict = {} # used for storing old data, so you don't need to specify all file info above, just new stuff

# with open('secrets.json', 'r', encoding='utf-8') as secs:
#     dict = json.load(secs)

# with open('secrets.json', 'w', encoding='utf-8') as secs:
#     z = dict | secrets

#     json.dump(z, secs, ensure_ascii=False, indent=4)

hash(QQlogin) = 7490910061332689875 
hash(QQpassword) = 3085619118335690969 


In [22]:
# QQlogin = ""
# QQpassword = ""

login_url = "https://forum.questionablequesting.com/login/login"
alerts_url = "https://forum.questionablequesting.com/account/alerts"

payload = {
    'login': QQlogin,
    'register': '0',
    'password': QQpassword,
    'cookie_check': '1',
    '_xfToken': '',
    'redirect': '/account/alerts'
}

userAgent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/105.0.0.0 Safari/537.36 Edg/105.0.1343.42"
loginHeaders = {
    "User-Agent": userAgent,
    'Referer': alerts_url,
    'Host': 'forum.questionablequesting.com',
    'Origin': 'https://forum.questionablequesting.com',
    #'Connection': 'keep-alive',
}

sessionCookieName = 'xf_session'
mceCookieName = 'xf_mce_ccv'

with requests.Session() as session:
    getRes = session.get(alerts_url)
    sessionCookie = getRes.cookies.get(sessionCookieName)

    print(f"getRes {sessionCookieName} = {sessionCookie}")

    loginHeaders['Cookie'] = f"{sessionCookieName}={sessionCookie}"

    print(f'loginHeaders = {loginHeaders}\n')

    print(f"{session.headers = }")
    for (header, value) in loginHeaders.items():
        session.headers[header] = value
    print(f"\n{session.headers = }\n")

    loginRes = session.post(login_url, data=payload)
    print(f'{session.cookies = }\nafter loginRes = session.post\n')

    session.close()

getRes xf_session = f9f975c2a5fa31f571168ce84f296be4
loginHeaders = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/105.0.0.0 Safari/537.36 Edg/105.0.1343.42', 'Referer': 'https://forum.questionablequesting.com/account/alerts', 'Host': 'forum.questionablequesting.com', 'Origin': 'https://forum.questionablequesting.com', 'Cookie': 'xf_session=f9f975c2a5fa31f571168ce84f296be4'}

session.headers = {'User-Agent': 'python-requests/2.28.2', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}

session.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/105.0.0.0 Safari/537.36 Edg/105.0.1343.42', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive', 'Referer': 'https://forum.questionablequesting.com/account/alerts', 'Host': 'forum.questionablequesting.com', 'Origin': 'https://forum.questionablequesting.com', 'Cookie': 'xf_

In [23]:
def convert_relative_date(relative_date: str) -> date:
    today = datetime.now().date()

    match relative_date:
        case "Today":
            return today
        case "Yesterday":
            return today - timedelta(days=1)
        case "Monday" | "Tuesday" | "Wednesday" | "Thursday" | "Friday" | "Saturday" | "Sunday":
            weekday = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"].index(relative_date.title())
            days_offset = (today.weekday() - weekday) % 7
            return today - timedelta(days=days_offset)
        case _ if datetime.strptime(relative_date, "%Y/%m/%d"):
            return datetime.strptime(relative_date, "%Y/%m/%d").date()
        case _ if datetime.strptime(relative_date, "%Y-%m-%d"):
            return datetime.strptime(relative_date, "%Y-%m-%d").date()
        case _:
            raise Exception()

# Example usage:
print(convert_relative_date("Today"), "Today")
print(convert_relative_date("Yesterday"), "Yesterday")
print("")

for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
    date = convert_relative_date(day)
    print(f"{date} {day}")

2023-06-02 Today
2023-06-01 Yesterday

2023-05-29 Monday
2023-05-30 Tuesday
2023-05-31 Wednesday
2023-06-01 Thursday
2023-06-02 Friday
2023-05-27 Saturday
2023-05-28 Sunday


In [25]:
from datetime import date

@dataclass
class AlertInfo:
    author_name: str | None = None
    author_link: str | None = None
    avatar_image_url: str | None = None
    post_link: str | None = None
    alert_text: str | None = None
    post_time: time | None = None
    # lengthy ergo likely a chapter post
    lengthy_response: bool = False
    post_date: date | None = None

    def to_json(self) -> str:
        json_dict = {
            'author_name': self.author_name,
            'author_link': self.author_link,
            'avatar_image_url': self.avatar_image_url,
            'post_link': self.post_link,
            'alert_text': self.alert_text,
            'lengthy_response': self.lengthy_response,
            'post_date': None if self.post_date is None else self.post_date.isoformat(),
            'post_time': None if self.post_time is None else self.post_time.strftime('%H:%M:%S')
        }
        return json.dumps(json_dict)

    @classmethod
    def from_json(cls, json_str: str) -> 'AlertInfo':
        json_dict = json.loads(json_str)
        post_date = None if json_dict['post_date'] is None else date.fromisoformat(json_dict['post_date'])
        post_time = None if json_dict['post_time'] is None else time.fromisoformat(json_dict['post_time'])
        return cls(
            author_name=json_dict['author_name'],
            author_link=json_dict['author_link'],
            avatar_image_url=json_dict['avatar_image_url'],
            post_link=json_dict['post_link'],
            alert_text=json_dict['alert_text'],
            lengthy_response=json_dict['lengthy_response'],
            post_date=post_date,
            post_time=post_time
        )

def print_alert_info(alert_info: AlertInfo) -> None:
    print("Alert Information:")
    print(f"- Author Name: {alert_info.author_name}")
    print(f"- Author Link: {alert_info.author_link}")
    print(f"- Avatar Image URL: {alert_info.avatar_image_url}")
    print(f"- Post Link: {alert_info.post_link}")
    print(f"- Alert Text: {alert_info.alert_text}")
    print(f"- Date: {alert_info.post_date}")
    print(f"- Time: {alert_info.post_time}")
    print(f"- Likely chapter: {alert_info.lengthy_response}")

def extract_alert_info(notif: Tag, url: URL = "https://forum.questionablequesting.com") -> AlertInfo:
    # Initialize the fields with None
    author_name = None
    author_link = None
    avatar_image_url = None
    post_link = None
    alert_text = None
    post_time = None

    # Extract author name and link
    author_tag = notif.find('a', class_='username subject')
    if author_tag is not None:
        author_name = author_tag.text.strip()
        author_link = f"{url}/{author_tag['href']}"

    # Extract author avatar image URL
    avatar_img = notif.find('img')
    if avatar_img is not None:
        avatar_image_url = f"{url}/{avatar_img['src']}"

    # Extract post link
    post_link_tag = notif.find('a', class_='PopupItemLink')
    if post_link_tag is not None:
        post_link = f"{url}/{post_link_tag['href']}"

    time_tag = notif.find('span', class_='time')
    if time_tag is not None:
        time_as_list = list([int(s) for s in time_tag.text.strip().split(':')])
        if len(time_as_list) in [2, 3]:
            post_time = time(time_as_list[0], time_as_list[1])

    # Extract alert text
    lengthy_response = False
    alert_text_div = notif.find('div', class_='alertText')
    if alert_text_div is not None:
        alert_text = alert_text_div.text.strip()
        alert_text = alert_text.replace('\n', ' ')

        # Remove the time from alert text
        if time_tag is not None:
            alert_text = alert_text.replace(time_tag.text.strip(), '')

        regex_pattern = r'replied with [0-9]+(\.[0-9]+)?k? words'
        lengthy_response = bool(re.search(regex_pattern, alert_text))

    # Return the extracted information as an instance of the AlertInfo data class
    return AlertInfo(
        author_name=author_name,
        author_link=author_link,
        avatar_image_url=avatar_image_url,
        post_link=post_link,
        alert_text=alert_text,
        post_time=post_time,
        lengthy_response=lengthy_response
    )

def time_it(f):
    @wraps(f)
    def wrapper():
        start = t.time_ns()
        ret = f()
        end = t.time_ns()
        print(f"{(end - start) / (1000 * 1000) = }")
        return ret
    return wrapper

@time_it
def get_alerts(html: str = testText) -> list[AlertInfo]:
    groups = strats._get_content_with_css_selector(testText, '.alertGroup')

    alerts = []
    for group in groups:
        date_text = group.select('h2.textHeading')[0].text
        date = convert_relative_date(date_text)

        notifs = group.select("li.primaryContent")
        for notif in notifs:
            alert_info = extract_alert_info(notif)
            alert_info.post_date = date
            alerts.append(alert_info)

    return alerts

alerts = get_alerts()
print(len(alerts))


(end - start) / (1000 * 1000) = 24.134641
30


In [105]:
no = datetime.now()
print(no)

datetime_string = no.strftime("%Y-%m-%d %H:%M:%S")
print(f"{datetime_string = }")

dt = datetime.strptime(datetime_string, "%Y-%m-%d %H:%M:%S")
print(f"{dt = }")

2023-06-02 19:41:37.498537
datetime_string = '2023-06-02 19:41:37'
dt = datetime.datetime(2023, 6, 2, 19, 41, 37)
